# 2. Разностные операторы и их спектры

Пусть на сетке записан профиль

$$
1,\;-1,\;1,\;-1,\;\ldots
$$

Значение меняется на каждом шаге. И всё же одна из самых распространённых формул для первой производной даст **ноль во всех узлах**. Как так получается? Начнём с того, как по трём соседним значениям найти первую и вторую производные.

## 1. Две производные по трём точкам

На отрезке от 0 до 1 возьмём равномерную периодическую сетку с шагом $h=1/N$. Пусть $U_k$ — значение функции в точке $x_k=kh$. Индексы идут по кругу, поэтому $U_N=U_0$.

**Первая производная** показывает наклон графика. Для её приближённого вычисления в точке $x_k$ проведём прямую через два соседних узла — левый и правый. Изменение высоты равно $U_{k+1}-U_{k-1}$, расстояние по горизонтали равно $2h$. Наклон этой прямой даёт **центральную разность** для первой производной:

$$
(D_0U)_k=\frac{U_{k+1}-U_{k-1}}{2h}.
$$

**Вторая производная** показывает, как меняется сам наклон. На отрезке слева от $x_k$ наклон равен $(U_k-U_{k-1})/h$, на отрезке справа — $(U_{k+1}-U_k)/h$. Середины этих отрезков находятся на расстоянии $h$. Поэтому

$$
(D_{xx}U)_k
=\frac{1}{h}\left(
\frac{U_{k+1}-U_k}{h}-\frac{U_k-U_{k-1}}{h}
\right)
=\frac{U_{k+1}-2U_k+U_{k-1}}{h^2}.
$$

На рисунке взят $u(x)=\cos(2\pi x)$. Вблизи вершины косинуса особенно хорошо видно различие между двумя формулами. Секущая через соседей вершины горизонтальна, поэтому первая разность равна нулю. При этом левый отрезок поднимается, а правый опускается. Наклон убывает, и вторая разность отрицательна.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as w
    from IPython.display import clear_output, display, Image
    from io import BytesIO
except ImportError:
    w = None

INK, TEAL, RUST = "#15334A", "#007C78", "#A84C26"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#A8B8BF", "axes.labelcolor": INK,
                     "text.color": INK, "font.size": 10, "axes.titlesize": 12,
                     "figure.facecolor": "white", "savefig.facecolor": "white"})


def number(value, digits=3):
    if abs(value) < 0.5 * 10**-digits:
        value = 0
    return (f"{value:.{digits}f}".rstrip("0").rstrip(".")
            .replace(".", ",").replace("-", "−"))


def experiment(figure, choices):
    defaults = {key: spec[-1] for key, spec in choices.items()}
    if w is None:
        figure(**defaults)
        plt.show()
        return
    controls = {}
    for key, (label, options, value) in choices.items():
        if isinstance(options, range):
            control = w.IntSlider(min=options.start, max=options[-1], value=value,
                                  step=options.step, continuous_update=False)
        else:
            control = w.Dropdown(options=options, value=value)
        control.description = label
        control.style.description_width = "initial"
        control.layout = w.Layout(width="310px", max_width="100%")
        controls[key] = control
    output = w.Output()

    def draw(change=None):
        fig = figure(**{key: control.value for key, control in controls.items()})
        with BytesIO() as buffer:
            fig.savefig(buffer, format="png", dpi=110, bbox_inches="tight", pad_inches=.1)
            img = Image(data=buffer.getvalue(), width=850)
        plt.close(fig)
        with output:
            clear_output(wait=True)
            display(img)

    for control in controls.values():
        control.observe(draw, names="value")
    layout = w.Layout(display="flex", flex_flow="row wrap")
    display(w.VBox([w.Box(list(controls.values()), layout=layout), output]))
    draw()


def stencil_figure(order=1, k=2):
    N, h = 12, 1/12
    neighbours = np.arange(k-1, k+2) * h
    vals = np.cos(2*np.pi*neighbours)
    left, right = np.diff(vals) / h
    # У узла 0 левый сосед изображён на продолжении периодического профиля.
    start = -h if k == 0 else 0
    x = np.linspace(start, 1, 600)
    nodes = np.arange(-1 if k == 0 else 0, N+1) * h
    fig, ax = plt.subplots(figsize=(8.4, 3.2), constrained_layout=True)
    ax.axhline(0, lw=.8, color="#C6D0D4")
    ax.plot(x, np.cos(2*np.pi*x), ":", color=INK, lw=1.4)
    ax.plot(nodes, np.cos(2*np.pi*nodes), "o", color=INK, ms=3)
    if order == 1:
        ax.plot(neighbours[[0, 2]], vals[[0, 2]], "-", lw=2.4, color=TEAL)
        note = "Наклон секущей  " + number((vals[2]-vals[0])/(2*h))
    else:
        ax.plot(neighbours[:2], vals[:2], "-", lw=2.4, color=TEAL)
        ax.plot(neighbours[1:], vals[1:], "--", lw=2.4, color=RUST)
        note = (f"Наклон слева  {number(left)}     ·     справа  {number(right)}"
                f"     ·     вторая разность  {number((right-left)/h)}")
    ax.plot(neighbours[0], vals[0], "s", color=TEAL, ms=6)
    ax.plot(neighbours[2], vals[2], "s", color=TEAL if order == 1 else RUST, ms=6)
    ax.plot(neighbours[1], vals[1], "o", mfc="white", mec=INK, mew=1.5, ms=6)
    ax.annotate(f"k = {k}", (neighbours[1], vals[1]), xytext=(0,8), ha="center",
                textcoords="offset points", color=INK)
    ax.set(xlim=(start-.02, 1.02), ylim=(-1.22,1.32), xlabel="Координата x",
           ylabel="Значение", xticks=[0,.25,.5,.75,1], yticks=[-1,0,1],
           title="Три соседние точки косинуса")
    ax.text(.5,-.30,note,transform=ax.transAxes,ha="center",va="center",fontsize=9)
    return fig


experiment(stencil_figure, {
    "order": ("Производная", [("Первая", 1), ("Вторая", 2)], 1),
    "k": ("Узел k", range(12), 2),
})

## 2. Как разностный оператор действует на гармонику?

Оператор $D_0$ берёт сразу весь вектор $U$ и возвращает новый вектор — значения первой разности во всех узлах. В каждом узле действует одно и то же правило. Например, для четырёх узлов его можно записать матрицей:

$$
D_0=\frac{1}{2h}
\begin{pmatrix}
0&1&0&-1\\
-1&0&1&0\\
0&-1&0&1\\
1&0&-1&0
\end{pmatrix}.
$$

В первой строке стоят коэффициенты при соседях узла $0$. Сосед справа имеет номер $1$, сосед слева — номер $3$. В каждой следующей строке коэффициенты сдвинуты на одну позицию вправо с переходом через край. Матрицу с таким устройством называют **циркулянтной**.

То же правило можно записать короче через оператор сдвига $S$. Пусть $(SU)_k=U_{k-1}$. Тогда обратный сдвиг даёт $(S^{-1}U)_k=U_{k+1}$, а единичный оператор $I$ оставляет $U$ без изменения. Наши две формулы принимают вид

$$
D_0=\frac{S^{-1}-S}{2h},
\qquad
D_{xx}=\frac{S^{-1}-2I+S}{h^2}.
$$

Теперь возьмём одну комплексную гармонику

$$
\varphi_k^{(n)}=e^{ik\theta_n},\qquad \theta_n=\frac{2\pi n}{N}.
$$

В соседнем узле справа её значение умножается на $e^{i\theta_n}$, а слева — на $e^{-i\theta_n}$. Поэтому для вычисления первой разности достаточно вычесть эти два множителя:

$$
(D_0\varphi^{(n)})_k
=\frac{e^{i\theta_n}-e^{-i\theta_n}}{2h}\,\varphi_k^{(n)}
=\underbrace{\frac{i\sin\theta_n}{h}}_{\mu_n}\,\varphi_k^{(n)}.
$$

Для второй разности таким же образом получаем

$$
(D_{xx}\varphi^{(n)})_k
=\frac{e^{i\theta_n}-2+e^{-i\theta_n}}{h^2}\,\varphi_k^{(n)}
=\underbrace{-\frac{4}{h^2}\sin^2\!\frac{\theta_n}{2}}_{\nu_n}\,\varphi_k^{(n)}.
$$

В обоих случаях получилась **та же гармоника, умноженная на число**. Если для ненулевого вектора $v$ выполнено $Dv=\lambda v$, то $v$ называют собственным вектором оператора $D$, а число $\lambda$ — его собственным значением. Здесь собственные векторы у двух операторов общие, а собственные значения разные.

## 3. Что показывает спектр?

**Спектром матрицы** называют множество её собственных значений. Для наших операторов весь спектр можно получить, подставляя в формулы номера гармоник. У $D_0$ числа $\mu_n$ лежат на мнимой оси. У $D_{xx}$ числа $\nu_n$ вещественны и неположительны.

Например, при $N=16$ гармоника с номером $n=4$ совершает четыре колебания на длине отрезка. Между соседними узлами её фаза меняется на $\theta_4=\pi/2$. Для неё первая разность даёт множитель $16i$, а вторая — $-512$. На рисунке соответствующее собственное значение отмечено ромбом.

In [2]:
def spectrum_figure(order=1, n=4):
    N, h = 16, 1/16
    modes = np.arange(-N//2, N//2)
    theta = 2*np.pi*modes/N
    values = (1j*np.sin(theta)/h if order == 1
              else -4*np.sin(theta/2)**2/h**2).astype(complex)
    values[np.abs(values) < 1e-10] = 0
    selected = values[modes == n][0]
    fig, ax = plt.subplots(figsize=(8.1, 3.45), constrained_layout=True)
    ax.axhline(0, lw=1, color="#9AACB4")
    ax.axvline(0, lw=1, color="#9AACB4")
    ax.plot(values.real, values.imag, "o", ms=5, color=TEAL)
    ax.plot(selected.real, selected.imag, "D", ms=10, mfc="none", mec=INK, mew=1.4)
    if order == 1:
        ax.set(xlim=(-39,39), ylim=(-18.5,18.5), yticks=[-16,-8,0,8,16])
        coefficient = number(selected.imag,4) + ("i" if selected.imag else "")
        symbol = r"\mu"
    else:
        ax.set(xlim=(-1065,45), ylim=(-205,205), xticks=[-1024,-768,-512,-256,0])
        coefficient = number(selected.real,4)
        symbol = r"\nu"
    ax.set_aspect("equal", adjustable="box")
    ax.set(xlabel="Действительная часть", ylabel="Мнимая часть",
           title="Спектр на сетке из 16 узлов")
    val = selected.imag if order == 1 else selected.real
    equality = "=" if np.isclose(val,round(val),atol=1e-10,rtol=0) else r"\approx"
    detail = "$" + rf"{symbol}_{{{n}}} {equality}" + "$ " + coefficient
    pairs = modes[np.isclose(values,selected,atol=1e-9,rtol=0) & (modes != n)]
    if pairs.size:
        detail += "     ·     То же значение при n = " + ", ".join(map(str,pairs))
    fig.supxlabel(detail,fontsize=9,color=INK)
    return fig


experiment(spectrum_figure, {
    "order": ("Оператор", [("Первая разность D₀", 1), ("Вторая разность Dₓₓ", 2)], 1),
    "n": ("Гармоника n", range(-8,8), 4),
})

Теперь видно, зачем знать спектр. Пусть профиль сложен из нескольких гармоник. Чтобы найти его первую разность, можно умножить коэффициент каждой гармоники на своё число $\mu_n$ и сложить результаты. Для второй разности вместо $\mu_n$ берётся $\nu_n$. Оператор действует на каждую составляющую независимо.

## 4. Насколько точно вычисляется производная?

У непрерывной гармоники $u(x)=e^{i\kappa x}$ обе производные известны точно:

$$
u'(x)=i\kappa\,u(x),\qquad u''(x)=-\kappa^2u(x).
$$

Здесь $\kappa=2\pi n$ — изменение фазы на единицу длины. За один шаг сетки фаза меняется на $\theta=\kappa h$.

Теперь можно сравнить множители разностных и точных производных одной и той же ненулевой гармоники. Для первой производной их отношение равно

$$
\frac{i\sin\theta/h}{i\kappa}=\frac{\sin\theta}{\theta}.
$$

Для второй производной получаем

$$
\frac{-4\sin^2(\theta/2)/h^2}{-\kappa^2}
=\left(\frac{\sin(\theta/2)}{\theta/2}\right)^2.
$$

У этих формул есть простое геометрическое прочтение. Если на одно колебание приходится $p$ шагов сетки, то за один шаг фаза меняется на $2\pi/p$ по модулю. При большом $p$ оба отношения близки к единице. А у гармоники из предыдущего примера всего **четыре шага на период**. Первая разность передаёт около $64\%$ амплитуды её точной производной, вторая — около $81\%$.

In [ ]:
def resolution_figure(p=4):
    x = np.linspace(0,1,401)
    nodes = np.arange(p+1) / p
    theta = 2*np.pi/p
    first = 0.0 if p == 2 else np.sin(theta)/theta
    second = (np.sin(theta/2)/(theta/2))**2
    ratios = 100*np.array([first,second])
    fig, (wave,bars) = plt.subplots(2,1,figsize=(8.4,4.0), constrained_layout=True,
                                     gridspec_kw={"height_ratios":[1.3,1]})
    wave.axhline(0,lw=.8,color="#C6D0D4")
    wave.plot(x,np.cos(2*np.pi*x),color=INK,lw=1.3)
    wave.plot(nodes,np.cos(2*np.pi*nodes),"o",color=TEAL,ms=4.5)
    wave.set(xlim=(-.015,1.015),ylim=(-1.18,1.18),xticks=[0,.5,1],yticks=[-1,0,1],
             xlabel="Доля периода", ylabel="Косинус",
             title=f"Шагов сетки на один период — {p}")
    bars.barh([1,0],[100,100],height=.34,color="#DDE5E7")
    bars.barh([1,0],ratios,height=.34,color=TEAL)
    for y, value in zip([1,0],ratios):
        bars.text(102,y,number(value,2)+" %",ha="left",va="center",fontsize=9)
    bars.set(xlim=(0,115), ylim=(-.5,1.5), xticks=[0,25,50,75,100], yticks=[1,0],
             yticklabels=["Первая, D₀","Вторая, Dₓₓ"],
             xlabel="Модуль разностного множителя относительно точного, %")
    return fig


experiment(resolution_figure, {"p": ("Шагов на период", range(2,33), 10)})

### Два шага на период

Вернёмся к профилю $U_k=(-1)^k$ на сетке с чётным числом узлов. Слева и справа от каждого узла стоит одно и то же число: $U_{k-1}=U_{k+1}=-U_k$. Подстановка в две разностные формулы даёт

$$
(D_0U)_k=0,\qquad
(D_{xx}U)_k=-\frac{4}{h^2}U_k.
$$

Это различие имеет наглядное следствие для **переноса**. Уравнение $u_t+c\,u_x=0$ с $c>0$ сдвигает профиль вправо со скоростью $c$. Если начальный профиль равен $\cos(\pi x/h)$, то через время $h/c$ он сдвинется на один узел и каждое узловое значение сменит знак. Если же заменить $u_x$ центральной разностью, получится $\dot U=-cD_0U=0$. Сеточный профиль останется неподвижным.

При **диффузии** $u_t=\chi u_{xx}$, где $\chi>0$, тот же косинус затухает. Точная вторая производная умножает его на $-\pi^2/h^2$, а разностная — на $-4/h^2$. Поэтому на сетке чередование тоже сглаживается, но коэффициент затухания составляет лишь $4/\pi^2\approx 0{,}405$ от точного.